# SLR-RAG : Step 1

## Package import

In [1]:
# Import packages
%load_ext autoreload
%autoreload 2
import pandas as pd
from slrrag.querier_scopus import query_scopus


pd.__version__

'3.0.3'

## Inputs

### Query

### Reference papers
Represent them as (TITLE,DOI)
The notebook will try to do the merge based on the DOI first, and on the title otherwise if the paper has no DOI.

In [2]:
from slrrag.main import get_new_reference_df

df_reference_paper = get_new_reference_df()
df_reference_paper

,title,doi,type,year,date,abstract
0,A dynamic risk assessment method to address sa...,10.1177/1475090219837409,NaN,<NA>,NaT,NaN
1,A Marine GIS-based Alert System to Prevent Ves...,10.51400/2709-6998.2462,NaN,<NA>,NaT,NaN
2,A Novel Method for Navigational Risk Assessmen...,10.1155/2021/4588333,NaN,<NA>,NaT,NaN
3,A System-Theory and Complex Network-Fused Appr...,10.3390/jmse11071306,NaN,<NA>,NaT,NaN
4,Allision risk analysis of offshore petroleum i...,10.1007/s13437-016-0123-7,NaN,<NA>,NaT,NaN
5,Assessing the impacts to vessel traffic from o...,NaN,NaN,<NA>,NaT,NaN
6,Assessing the validity of navigation risk asse...,10.1016/j.ocecoaman.2022.106078,NaN,<NA>,NaT,NaN
7,Evaluation of Ship Importance in Offshore Wind...,10.3390/su17188252,NaN,<NA>,NaT,NaN
8,Improving the coexistence of offshore wind far...,10.1007/s13437-018-0149-0,NaN,<NA>,NaT,NaN
9,Likelihood of a marine vessel accident from wi...,10.1002/we.1935,NaN,<NA>,NaT,NaN


## Query databases

### Add scopus papers

In [3]:
from slrrag.main import add_is_reference_column, NAME_COL_IS_REFERENCE

main_query = """
(
    (
        ("offshore") 
        AND 
        ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR  "windmill" OR 
        "wind turbine" OR "wind park" OR "wind energy" OR "platform")
    )
    OR 
    ("flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")
)
AND
(
	( "collision" OR "allision" OR "near miss" OR "near-miss" OR "risk" OR "safety" OR "safe") AND 
    ( "navigate" OR "navigation" OR "navigational" OR "maritime" OR "shipping" OR "traffic")
	AND 
	("quantitative" OR "quantification" OR "probability" OR "probabilistic" OR "likelihood" 
        OR "frequency" OR "assess" OR "assessment" OR "predict" OR "predicting"
        OR "prediction" OR "model" OR "modeling" OR "modelling" OR "simulate"
        OR "simulation" OR "estimate" OR "estimation" OR "detect" OR "detection")
)
"""

# You can edit your scopus API key in ~/.config/pybliometrics.cfg if needed

is_dry_run = False
df_scopus = query_scopus(main_query, dry_run=is_dry_run, refresh=True)
if not is_dry_run:
    # Check if reference papers are present from previous query on scopus
    df_scopus = add_is_reference_column(df_scopus)
    print("---\nReference papers in Scopus : ", df_scopus[NAME_COL_IS_REFERENCE].sum())

941 found in Scopus.
TITLE-ABS-KEY( 
(
    (
        ("offshore") 
        AND 
        ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR  "windmill" OR 
        "wind turbine" OR "wind park" OR "wind energy" OR "platform")
    )
    OR 
    ("flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")
)
AND
(
	( "collision" OR "allision" OR "near miss" OR "near-miss" OR "risk" OR "safety" OR "safe") AND 
    ( "navigate" OR "navigation" OR "navigational" OR "maritime" OR "shipping" OR "traffic")
	AND 
	("quantitative" OR "quantification" OR "probability" OR "probabilistic" OR "likelihood" 
        OR "frequency" OR "assess" OR "assessment" OR "predict" OR "predicting"
        OR "prediction" OR "model" OR "modeling" OR "modelling" OR "simulate"
        OR "simulation" OR "estimate" OR "estimation" OR "detect" OR "detection")
)
)
":


100%|██████████| 38/38 [00:40<00:00,  1.09s/it]

---
Reference papers in Scopus :  17


In [4]:
from slrrag.main import merge_papers

df_papers_01 = merge_papers(df_reference_paper, df_scopus, "scopus")
df_papers_01

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,is_reference
0,10th International Multidisciplinary Scientifi...,NaN,other,<NA>,2010-12-01,The proceedings contain 320 papers. The specia...,2-s2.0-84890693475,NaN,939,False
1,10th International Multidisciplinary Scientifi...,NaN,other,<NA>,2010-12-01,The proceedings contain 320 papers. The specia...,2-s2.0-84890753233,NaN,938,False
2,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886697517,NaN,405,False
3,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886707330,NaN,410,False
4,"12th AIAA Aviation Technology, Integration, an...",NaN,other,<NA>,2012-12-01,The proceedings contain 306 papers. The specia...,2-s2.0-84880934476,NaN,347,False
...,...,...,...,...,...,...,...,...,...,...
939,WIND TURBINE GENERATOR IMPACTS TO MARINE VESSE...,10.17226/26430,book,<NA>,2022-01-01,Offshore wind energy development is poised to ...,2-s2.0-85138382434,NaN,93,False
940,WindNet: A Mobile Base Station Infrastructure ...,10.1109/IHTC61819.2024.10855097,conference-paper,<NA>,2024-01-01,"The maritime industry, encompassing sectors su...",2-s2.0-85217840303,5G | Communication Network | Maritime | Mobile...,102,False
941,Worker health and safety on maritime platforms...,10.1590/1413-81232026318.04282024,article,<NA>,2026-01-01,Maritime platforms pose threats to the health ...,2-s2.0-105048874749,Ergology | Offshore | Safety and health at wor...,343,False
942,Year-round shipping in the Canadian Arctic: Ic...,NaN,conference-paper,<NA>,2007-01-01,Extensive drilling programs took place in the ...,2-s2.0-84901373244,NaN,749,False


### Add OpenAlex papers

You will need the API key of

Make sure you have the API key of OpenAlex setup.

In [5]:
from slrrag.querier_openalex import query_openalex

df_openalex = query_openalex(main_query, dry_run=is_dry_run)
df_openalex

Query : 
(
    (
        ("offshore") 
        AND 
        ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR  "windmill" OR 
        "wind turbine" OR "wind park" OR "wind energy" OR "platform")
    )
    OR 
    ("flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")
)
AND
(
	( "collision" OR "allision" OR "near miss" OR "near-miss" OR "risk" OR "safety" OR "safe") AND 
    ( "navigate" OR "navigation" OR "navigational" OR "maritime" OR "shipping" OR "traffic")
	AND 
	("quantitative" OR "quantification" OR "probability" OR "probabilistic" OR "likelihood" 
        OR "frequency" OR "assess" OR "assessment" OR "predict" OR "predicting"
        OR "prediction" OR "model" OR "modeling" OR "modelling" OR "simulate"
        OR "simulation" OR "estimate" OR "estimation" OR "detect" OR "detection")
)



1338 found in Openalex.


 97%|█████████▋| 26/26.76 [00:25<00:00,  1.05it/s]/home/vlitoux/Documents/SLR-RAG/.venv/lib/python3.12/site-packages/tqdm/std.py:635: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
101%|██████████| 27/26.76 [00:26<-00:00,  1.03it/s]


,doi,id_openalex,title,date,abstract,type,year,relevance_openalex,language_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex
0,10.1016/j.ress.2020.107086,https://openalex.org/W3039653437,Realising advanced risk assessment of vessel t...,2020-07-02,NaN,article,<NA>,482.061130,en,https://content.openalex.org/works/W3039653437...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Offshore wind power|Turbine|Bayesian network|C...
1,10.1111/j.1474-919x.2006.00510.x,https://openalex.org/W2142021246,Information needs to support environmental imp...,2006-03-01,European legislation requires Strategic Enviro...,article,<NA>,421.708600,en,https://content.openalex.org/works/W2142021246...,Avian ecology and behavior,Environmental Science,Ecology,Physical Sciences,Habitat|Environmental resource management|Envi...
2,10.1016/j.ocecoaman.2022.106078,https://openalex.org/W4211011661,Assessing the validity of navigation risk asse...,2022-02-09,The developments of offshore wind farms can pl...,article,<NA>,370.358640,en,https://content.openalex.org/works/W4211011661...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Risk assessment|Risk analysis (engineering)|Cr...
3,10.1115/1.4025030,https://openalex.org/W1989654646,Methodology for Wind/Wave Basin Testing of Flo...,2014-03-22,Scale-model wave basin testing is often employ...,article,<NA>,343.631600,en,https://content.openalex.org/works/W1989654646...,Wave and Wind Energy Systems,Engineering,Ocean Engineering,Physical Sciences,Turbine|Offshore wind power|Marine engineering...
4,10.1016/j.oceaneng.2018.01.008,https://openalex.org/W2793925100,Probabilistic approach for collision risk anal...,2018-02-13,NaN,article,<NA>,322.455870,en,https://content.openalex.org/works/W2793925100...,Structural Integrity and Reliability Analysis,Engineering,Mechanical Engineering,Physical Sciences,Collision|Probabilistic logic|Submarine pipeli...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1333,10.1177/0740277511402793,https://openalex.org/W2107709274,New Capitols of Capital,2011-03-01,Since the beginning of the global trade in sec...,article,<NA>,0.229913,en,https://content.openalex.org/works/W2107709274...,Global Urban Networks and Dynamics,Social Sciences,Urban Studies,Social Sciences,Pavilion|Treasure|Capital (architecture)|Econo...
1334,10.1093/tandt/ttr145,https://openalex.org/W2327172936,The missionary position in trust jurisprudence,2012-01-01,If the Franciscan friars had beaten the Jesuit...,article,<NA>,0.224357,en,https://content.openalex.org/works/W2327172936...,Religious Freedom and Discrimination,Social Sciences,Law,Social Sciences,Jurisprudence|Position (finance)|Political sci...
1335,10.4324/9780203014288-28,https://openalex.org/W4233274104,O,2006-03-21,POLLUTION atmospheric obscurity \nobservateur ...,other,<NA>,0.189484,fr,https://content.openalex.org/works/W4233274104...,NaN,NaN,NaN,NaN,Materials science
1336,10.1108/978-1-78973-545-120191001,https://openalex.org/W4241791061,Index,2019-08-06,"Citation (2019), ""Index"", Oseni, U.A., Hassan,...",other,<NA>,0.179114,en,https://content.openalex.org/works/W4241791061...,"Linguistic, Cultural, and Literary Studies",Social Sciences,Sociology and Political Science,Social Sciences,Money laundering|Business|Publishing|Index (ty...


In [6]:
from slrrag.main import merge_papers

df_papers_02 = merge_papers(df_papers_01, df_openalex, "openalex")
df_papers_02

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,is_reference,id_openalex,relevance_openalex,language_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex
0,10th International Multidisciplinary Scientifi...,NaN,other,<NA>,2010-12-01,The proceedings contain 320 papers. The specia...,2-s2.0-84890693475,NaN,939,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10th International Multidisciplinary Scientifi...,NaN,other,<NA>,2010-12-01,The proceedings contain 320 papers. The specia...,2-s2.0-84890753233,NaN,938,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886697517,NaN,405,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886707330,NaN,410,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"12th AIAA Aviation Technology, Integration, an...",NaN,other,<NA>,2012-12-01,The proceedings contain 306 papers. The specia...,2-s2.0-84880934476,NaN,347,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1758,《Quantum Autonomous Infrastructure at the Limi...,10.5281/zenodo.22981316,other,<NA>,2026-09-26,《Quantum Autonomous Infrastructure at the Limi...,NaN,NaN,<NA>,False,https://openalex.org/W7214481092,4.351813,NaN,https://content.openalex.org/works/W7214481092...,NaN,NaN,NaN,NaN,Autonomy|Action (physics)|Quality (philosophy)...
1759,混和材を用いて蒸気養生したコンクリートの寒冷地環境における塩害に関する実験的研究,10.14943/doctoral.k15679,dissertation,<NA>,2023-12-25,"In the cold regions of Japan, the infrastructu...",NaN,NaN,<NA>,False,https://openalex.org/W7207587084,0.483604,en,https://content.openalex.org/works/W7207587084...,Concrete and Cement Materials Research,Engineering,Civil and Structural Engineering,Physical Sciences,Precast concrete|Durability|Corrosion|Cementit...
1760,넙치 양식장 위해요소중점관리기준(HACCP) 적용모델 개발,NaN,article,<NA>,2013-10-01,We attempted to apply the HACCP system adopted...,NaN,NaN,<NA>,False,https://openalex.org/W1786544126,1.642470,ko,https://content.openalex.org/works/W1786544126...,Technology and Data Analysis,Computer Science,Information Systems,Physical Sciences,Critical control point|Hazard analysis and cri...
1761,서남해 해상풍력발전단지 내 항로형태에 따른 선박통항 위험도 평가에 관한 연구,NaN,article,<NA>,2019-10-31,Domestic southwest offshore wind farms have pr...,NaN,NaN,<NA>,False,https://openalex.org/W3088764835,4.339668,ko,https://content.openalex.org/works/W3088764835...,Marine and Coastal Research,Engineering,Ocean Engineering,Physical Sciences,Submarine pipeline|Traffic volume|Offshore win...


### Add dimensions

In [7]:
from slrrag.main import print_dimensions_export
from slrrag.query_converter import query_to_dimensions_url
from slrrag.settings import PATH_DIMENSIONS_EXPORT


dimensions_url = query_to_dimensions_url(main_query)
export_path_dimensions = PATH_DIMENSIONS_EXPORT
print_dimensions_export(dimensions_url, export_path_dimensions)


Dimensions.ai URL:
https://app.dimensions.ai/discover/publication?search_mode=content&search_text=%0A(%0A%20%20%20%20(%0A%20%20%20%20%20%20%20%20(%22offshore%22)%20%0A%20%20%20%20%20%20%20%20AND%20%0A%20%20%20%20%20%20%20%20(%22infrastructure%22%20OR%20%22construction%22%20OR%20%22installation%22%20OR%20%22windfarm%22%20OR%20%22wind%20farm%22%20OR%20%20%22windmill%22%20OR%20%0A%20%20%20%20%20%20%20%20%22wind%20turbine%22%20OR%20%22wind%20park%22%20OR%20%22wind%20energy%22%20OR%20%22platform%22)%0A%20%20%20%20)%0A%20%20%20%20OR%20%0A%20%20%20%20(%22flotel%22%20OR%20%22Aquaculture%20farm%22%20OR%20%22Artificial%20Island%22%20OR%20%22Synthetic%20Island%22)%0A)%0AAND%0A(%0A(%20%22collision%22%20OR%20%22allision%22%20OR%20%22near%20miss%22%20OR%20%22near-miss%22%20OR%20%22risk%22%20OR%20%22safety%22%20OR%20%22safe%22)%20AND%20%0A%20%20%20%20(%20%22navigate%22%20OR%20%22navigation%22%20OR%20%22navigational%22%20OR%20%22maritime%22%20OR%20%22shipping%22%20OR%20%22traffic%22)%0AAND%20%0A(%22q

In [8]:
export_path_dimensions

PosixPath('/home/vlitoux/Documents/SLR-RAG/data/papers_selection/dimensions-export.csv')

In [11]:
from slrrag.querier_dimensions import read_dimensions_export

df_dimensions = read_dimensions_export(export_path_dimensions)
df_dimensions

,doi,title,date,abstract,type,year,id_dimensions,topic_dimensions,keywords_dimensions
0,10.1016/j.marpolbul.2014.06.004,Getting it right for the North Atlantic right ...,2014-07-04,The North Atlantic right whale (Eubalaena glac...,article,<NA>,https://app.dimensions.ai/details/publication/...,"C13 Architecture, Built Environment and Planning",Animal Migration; Animals; Atlantic Ocean; Eco...
1,10.1038/s41598-023-36114-3,Analysis of navigational risk indicators as a ...,2023-06-07,This study concerns the analysis of navigation...,article,<NA>,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
2,10.3390/s20051246,Safe Navigation Distance Between Marine Routes...,2020-02-25,The purpose of this study was to determine the...,article,<NA>,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
3,10.1016/j.marpolbul.2025.118582,Identification of oil spills in the eastern Ar...,2025-08-15,Oil spills pose severe ecological and socio-ec...,article,<NA>,https://app.dimensions.ai/details/publication/...,B07 Earth Systems and Environmental Sciences,Petroleum Pollution; Environmental Monitoring;...
4,10.1016/j.heliyon.2024.e26788,The energy park of the future: Modelling the c...,2024-02-28,"To mitigate the effects of climate change, a s...",article,<NA>,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
...,...,...,...,...,...,...,...,...,...
858,10.65154/jmst.1099,THỰC HIỆN CHIẾN LƯỢC CẮT ĐỈNH TẢI DỰA TRÊN THU...,2026-07-30,Bài báo nghiên cứu chiến lược cắt đỉnh tải nhằ...,article,<NA>,https://app.dimensions.ai/details/publication/...,NaN,NaN
859,10.4043/19918-ms,SS: Testing in the Offshore: An Experimental S...,2009-05-04,Abstract This paper presents and discusses the...,conference-proceeding,<NA>,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
860,10.5204/mcj.3251,The Emergence of the Shallow State and the Cap...,2026-04-21,Introduction: A Shift Toward the ‘Dark Enlight...,article,<NA>,https://app.dimensions.ai/details/publication/...,C19 Politics and International Studies,NaN
861,10.51952/9781447377740,"Disrupting Anti-Trafficking Norms, Survivors, ...",2026-09-10,This timely and critical volume calls for a fu...,book,<NA>,https://app.dimensions.ai/details/publication/...,C20 Social Work and Social Policy,NaN


In [12]:
df_papers_03 = merge_papers(df_papers_02, df_dimensions, "dimensions")
df_papers_03

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,is_reference,...,language_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex,id_dimensions,topic_dimensions,keywords_dimensions
0,1 Probability of ship collision and grounding,10.1016/b978-0-12-815022-1.00001-3,book-chapter,<NA>,NaT,"To quantify the risk involved in ship traffic,...",NaN,NaN,<NA>,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
1,10th International Multidisciplinary Scientifi...,NaN,other,<NA>,2010-12-01,The proceedings contain 320 papers. The specia...,2-s2.0-84890693475,NaN,939,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10th International Multidisciplinary Scientifi...,NaN,other,<NA>,2010-12-01,The proceedings contain 320 papers. The specia...,2-s2.0-84890753233,NaN,938,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886697517,NaN,405,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10th International Symposium on Practical Desi...,NaN,other,<NA>,2007-12-01,The proceedings contain 165 papers. The specia...,2-s2.0-84886707330,NaN,410,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1911,水域環境のマクロ的評価手法に関する研究 : ボロノイメディアルアキスを利用して,10.9749/jin.100.229,article,<NA>,NaT,Large scale marine constructions such as an of...,NaN,NaN,<NA>,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN
1912,混和材を用いて蒸気養生したコンクリートの寒冷地環境における塩害に関する実験的研究,10.14943/doctoral.k15679,dissertation,<NA>,2023-12-25,"In the cold regions of Japan, the infrastructu...",NaN,NaN,<NA>,False,...,en,https://content.openalex.org/works/W7207587084...,Concrete and Cement Materials Research,Engineering,Civil and Structural Engineering,Physical Sciences,Precast concrete|Durability|Corrosion|Cementit...,NaN,NaN,NaN
1913,넙치 양식장 위해요소중점관리기준(HACCP) 적용모델 개발,NaN,article,<NA>,2013-10-01,We attempted to apply the HACCP system adopted...,NaN,NaN,<NA>,False,...,ko,https://content.openalex.org/works/W1786544126...,Technology and Data Analysis,Computer Science,Information Systems,Physical Sciences,Critical control point|Hazard analysis and cri...,NaN,NaN,NaN
1914,서남해 해상풍력발전단지 내 항로형태에 따른 선박통항 위험도 평가에 관한 연구,NaN,article,<NA>,2019-10-31,Domestic southwest offshore wind farms have pr...,NaN,NaN,<NA>,False,...,ko,https://content.openalex.org/works/W3088764835...,Marine and Coastal Research,Engineering,Ocean Engineering,Physical Sciences,Submarine pipeline|Traffic volume|Offshore win...,NaN,NaN,NaN


### Add useful columns

In [13]:
from slrrag.main import complete_papers_df, show_merged_stats
from slrrag.settings import PATH_CSV_01_QUERY_RESULTS
from slrrag.toolbox.files import slugify

databases_names = ["scopus", "openalex", "dimensions"]

df_papers_04 = complete_papers_df(df_papers_03, database_names=databases_names)
df_papers_04["title_slug"] = df_papers_04["title"].apply(slugify)
show_merged_stats(df_papers_04, databases_names)

---------CLEANING------------
Removed 133 duplicates
---------SUMMARIZING------------
Total number of papers found: 1783
Reference papers found: 19 over 19
Number of common papers in all databases: 483
Number of papers in scopus: 915
Number of reference papers in scopus: 17
Number of papers in openalex: 1259
Number of reference papers in openalex: 19
Number of papers in dimensions: 857
Number of reference papers in dimensions: 17


In [14]:
from slrrag.toolbox import query_treemap

tree = query_treemap.parse_query(main_query)
agg = query_treemap.aggregate_by_node(
    df=df_papers_04,
    tree=tree,
    columns=["title", "abstract"],
    binary_columns=["is_reference"],
    stemming=True,
).sort_values("count", ascending=False)
fig = query_treemap.treemap(agg, width=1250, height=750)
fig

#### Add from first choice steps

In [15]:
from slrrag.settings import PATH_INPUT
from slrrag.toolbox.files import slugify

df_selected_papers_round_1 = pd.read_csv(
    PATH_INPUT / "nra_kept_papers_5_without_bridge_snowballed.csv"
)
df_selected_papers_round_1["title_slug"] = df_selected_papers_round_1["title"].apply(
    slugify
)
df_selected_papers_round_1["is_chosen_first_nra"] = True
# Add is_reference column
df_papers_05 = (
    pd.merge(
        df_selected_papers_round_1[["title_slug", "is_chosen_first_nra"]],
        df_papers_04,
        on="title_slug",
        how="outer",
    )
    .fillna({"is_reference": False, "is_found": False, "is_chosen_first_nra": False})
    .astype({"is_found": bool, "is_chosen_first_nra": bool})
)
# df_selected_papers_round_1[
#     ~(df_selected_papers_round_1["title_slug"].isin(df_papers_04["title_slug"])) | ~(df_selected_papers_round_1["is_found"])
# ]
df_papers_05.groupby(["is_chosen_first_nra", "is_found"]).size()

is_chosen_first_nra  is_found
False                True        1734
True                 True          49
dtype: int64

In [16]:
list_missing_papers = list(
    df_papers_05[(df_papers_05["is_chosen_first_nra"]) & ~(df_papers_05["is_found"])]
    .sort_values(by="is_reference", ascending=False)
    .apply(
        lambda x: (
            x.loc["title_slug"] + "  (Ref)"
            if x.loc["is_reference"]
            else x.loc["title_slug"]
        ),
        axis=1,
        result_type="reduce",
    )
)

print("Numbers missing papers : ", len(list_missing_papers))

print("\n".join(list_missing_papers))

Numbers missing papers :  0



In [17]:
df_papers_05.to_csv(PATH_CSV_01_QUERY_RESULTS, index=False)
print("Phase 1 results saved to :", PATH_CSV_01_QUERY_RESULTS)

Phase 1 results saved to : /home/vlitoux/Documents/SLR-RAG/data/papers_selection/01_papers_query_results.csv


## Papers Analysis

#### Upset plot

In [18]:
import pandas as pd

from slrrag.settings import PATH_CSV_01_QUERY_RESULTS

df_papers = pd.read_csv(PATH_CSV_01_QUERY_RESULTS)

df_papers.dtypes

title_slug                    str
is_chosen_first_nra          bool
title                         str
doi                           str
type                          str
year                      float64
date                          str
abstract                      str
id_scopus                     str
keywords_scopus               str
relevance_order_scopus    float64
is_reference                 bool
id_openalex                   str
relevance_openalex        float64
language_openalex             str
pdf                           str
topic_openalex                str
fields_openalex               str
subfields_openalex            str
domain_openalex               str
keywords_openalex             str
id_dimensions                 str
topic_dimensions              str
keywords_dimensions           str
in_scopus                    bool
in_openalex                  bool
in_dimensions                bool
is_found                     bool
dtype: object

In [19]:
from slrrag.settings import PATH_UPSET_DISTINCT
from slrrag.toolbox.plotly_visualze import create_upset_plot

databases_names = ["scopus", "openalex", "dimensions"]

path_upset_distinct = PATH_UPSET_DISTINCT
fig_upset_distinct = create_upset_plot(
    df_papers, ["in_" + x for x in databases_names] + ["is_reference"], mode="distinct"
)
fig_upset_distinct.figure.write_html(path_upset_distinct)
print("Upset distinct plot saved to : ", path_upset_distinct)
fig_upset_distinct.figure

Upset distinct plot saved to :  /home/vlitoux/Documents/SLR-RAG/data/papers_selection/upset_distinct.html


In [20]:
from slrrag.settings import PATH_UPSET_INTERSECT

path_upset_intersect = PATH_UPSET_INTERSECT
fig_upset_distinct = create_upset_plot(
    df_papers, ["in_" + x for x in databases_names] + ["is_reference"], mode="intersect"
)
fig_upset_distinct.figure.write_html(path_upset_intersect)
print("Upset intersect plot saved to : ", path_upset_intersect)
fig_upset_distinct.figure

Upset intersect plot saved to :  /home/vlitoux/Documents/SLR-RAG/data/papers_selection/upset_intersect.html


## Old : with lens

In [ ]:
from slrrag.main import print_lens_export_info
from slrrag.query_converter import convert_query_to_lens
from slrrag.settings import PATH_LENS_EXPORT

lens_export_file = PATH_LENS_EXPORT
lens_query = convert_query_to_lens(main_query)
print_lens_export_info(lens_query, lens_export_file)


Lens query:

(
    (
        ((title:("offshore") OR abstract:("offshore"))) 
        AND 
        ((title:("infrastructure") OR abstract:("infrastructure")) OR (title:("construction") OR abstract:("construction")) OR (title:("installation") OR abstract:("installation")) OR (title:("windfarm") OR abstract:("windfarm")) OR (title:("wind farm") OR abstract:("wind farm")) OR  (title:("windmill") OR abstract:("windmill")) OR 
        (title:("wind turbine") OR abstract:("wind turbine")) OR (title:("wind park") OR abstract:("wind park")) OR (title:("wind energy") OR abstract:("wind energy")) OR (title:("platform") OR abstract:("platform")) OR (title:("quay") OR abstract:("quay")) OR (title:("wharf") OR abstract:("wharf")) OR (title:("lidar") OR abstract:("lidar")))
    )
    OR 
    (((title:("bridge") OR abstract:("bridge")) AND NOT (title:("bridge the gap") OR abstract:("bridge the gap")) AND NOT (title:("bridge therapy") OR abstract:("bridge therapy"))) OR (title:("flotel") OR abstract:

In [ ]:
from slrrag.querier_lens import read_lens_export

df_lens = read_lens_export(lens_export_file)
df_lens

,doi,title,date,abstract,type,year,id_lens,topic_lens,keywords_lens
0,10.1016/j.apor.2025.104564,Multi-perspective nautical safety risk assessm...,NaT,"The increasing amount of activities at sea, in...",article,<NA>,017-608-973-984-054,Offshore wind power; Submarine pipeline; Persp...,NaN
1,NaN,Maritime engineering and technology : proceedi...,NaT,Ship traffic Maritime situational awareness in...,book,<NA>,080-703-447-465-297,Engineering; Marine engineering; Marine techno...,NaN
2,10.1115/1.4053700,Profiles of Two JOMAE Associate Editors (The F...,2022-02-18,"For 35 years, beginning in 1987, the ASME Jour...",article,<NA>,174-185-838-473-498,China; Associate editor; Library science; Poli...,NaN
3,10.1002/qre.3611,Advanced reliability and safety methodologies ...,2024-06-28,Modern engineering systems tend to be more com...,article,<NA>,187-502-147-124-430,Reliability engineering; Reliability (semicond...,NaN
4,10.3390/su17188252,Evaluation of Ship Importance in Offshore Wind...,2025-09-14,<jats:p>With the rapid expansion of offshore w...,article,<NA>,196-025-595-838-127,Offshore wind power; Submarine pipeline; Marin...,NaN


In [ ]:
df_papers = merge_papers(df_papers, df_lens, "lens")
df_papers

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,is_reference,...,topic_dimensions,keywords_dimensions,in_scopus,in_openalex,in_dimensions,is_found,title_slug,id_lens,topic_lens,keywords_lens
0,1 Probability of ship collision and grounding,10.1016/b978-0-12-815022-1.00001-3,book-chapter,<NA>,NaT,"To quantify the risk involved in ship traffic,...",NaN,NaN,NaN,False,...,B12 Engineering,NaN,False,False,True,True,1-probability-of-ship-collision-and-grounding,NaN,NaN,NaN
1,1915 Çanakkale Bridge – Meeting the Challenge,10.1007/978-3-030-59169-4_4,book-chapter,2021,2021-04-28,The 1915 Çanakkale Bridge is a world record 20...,NaN,NaN,NaN,False,...,B12 Engineering,NaN,False,False,True,True,1915-çanakkale-bridge-meeting-the-challenge,NaN,NaN,NaN
2,2 External dynamics of ship collisions and gro...,10.1016/b978-0-12-815022-1.00002-5,book-chapter,<NA>,NaT,"This chapter presents analytical, closed-form ...",NaN,NaN,NaN,False,...,B12 Engineering,NaN,False,False,True,True,2-external-dynamics-of-ship-collisions-and-gro...,NaN,NaN,NaN
3,3 Internal mechanics of ship collision and gro...,10.1016/b978-0-12-815022-1.00003-7,book-chapter,<NA>,NaT,This chapter gives simplified analytical metho...,NaN,NaN,NaN,False,...,B12 Engineering,NaN,False,False,True,True,3-internal-mechanics-of-ship-collision-and-gro...,NaN,NaN,NaN
4,34 Risk and system integrity level assessment ...,10.1016/b978-0-12-818746-3.00014-2,book-chapter,<NA>,NaT,Offshore manufacturing facilities include dril...,NaN,NaN,NaN,False,...,B12 Engineering,NaN,False,False,True,True,34-risk-and-system-integrity-level-assessment-...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2006,船舶の衝突に対するペルプレン緩衝工の動的特性について,10.9749/jin.94.59,article,<NA>,NaT,Owing to increase of marine traffic and bridge...,NaN,NaN,NaN,False,...,B12 Engineering,NaN,False,False,True,True,船舶の衝突に対するペルプレン緩衝工の動的特性について,NaN,NaN,NaN
2007,船舶の衝突に対するロープ式緩衝工の効果に関する基礎研究,10.9749/jin.73.111,article,<NA>,NaT,"Recently, ship collision accidents with bridge...",NaN,NaN,NaN,False,...,B12 Engineering,NaN,False,False,True,True,船舶の衝突に対するロープ式緩衝工の効果に関する基礎研究,NaN,NaN,NaN
2008,船舶衝突事故例からみた橋梁付近の水路の安全性,10.9749/jin.84.103,article,<NA>,NaT,"In these years, many bridges which span the wa...",NaN,NaN,NaN,False,...,B12 Engineering,NaN,False,False,True,True,船舶衝突事故例からみた橋梁付近の水路の安全性,NaN,NaN,NaN
2009,避航操船における操船情報と操船者行動の関係,10.9749/jin.113.9,article,<NA>,NaT,"Using the ship-handling simulator, mariners' c...",NaN,NaN,NaN,False,...,B12 Engineering,NaN,False,False,True,True,避航操船における操船情報と操船者行動の関係,NaN,NaN,NaN
